# 01 - Resolucao de Problemas Complexos em Fisica
> Use o Claude para auxiliar na formulacao e verificacao de solucoes em mecanica quantica, termodinamica e eletromagnetismo

**Modulo:** `10_engfis_academico` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


## Setup e Configuracao

Carregamos a SDK da Anthropic e criamos uma funcao auxiliar `ask()` para facilitar as chamadas.

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

## Claude como Tutor de Fisica

Vamos configurar um **system prompt** que transforma o Claude em um professor de fisica.
Ele deve explicar conceitos passo a passo, usar notacao matematica e verificar unidades.

In [ ]:
SYSTEM_TUTOR = """
Voce e um professor de Fisica da UFRGS, especialista em mecanica quantica,
termodinamica e eletromagnetismo. Ao resolver problemas:
1. Identifique as grandezas conhecidas e desconhecidas
2. Escolha a lei ou principio fisico adequado
3. Desenvolva a solucao passo a passo, justificando cada etapa
4. Verifique unidades e ordem de grandeza do resultado
5. Faca um comentario fisico sobre o significado do resultado
Use notacao LaTeX para equacoes quando apropriado.
""".strip()

### Exemplo: Calculo de Entropia em Expansao de Gas Ideal

Um mol de gas ideal se expande isotermicamente de V1 = 10 L para V2 = 30 L a T = 300 K.
Qual a variacao de entropia?

In [ ]:
problema_termo = """
Problema de Termodinamica:
Um mol de gas ideal monoatomico sofre uma expansao isotermica reversivel
de V1 = 10 L para V2 = 30 L, a temperatura T = 300 K.

Calcule:
a) A variacao de entropia do gas (Delta S)
b) O calor trocado com o reservatorio
c) O trabalho realizado pelo gas

Explique cada passo detalhadamente.
"""

resposta = ask(problema_termo, system=SYSTEM_TUTOR, model=SONNET, max_tokens=2048)
print(resposta)

**Observacao:** Note como o Claude identifica que para um processo isotermico reversivel,
`Delta S = nR ln(V2/V1)`, verifica as unidades (J/K) e comenta que a entropia aumenta
porque o gas ocupa um volume maior (mais microestados acessiveis).

---

## Mecanica Quantica: Verificacao de Solucoes

Uma tecnica poderosa e enviar ao Claude uma solucao **com erros propositais**
e pedir que ele identifique os problemas. Isso treina o raciocinio critico.

In [ ]:
solucao_errada = """
Problema: Particula em uma caixa unidimensional de largura L.
Encontre as autoenergias e autofuncoes.

Minha solucao (verifique se esta correta):

1. A equacao de Schrodinger independente do tempo e:
   -(hbar^2 / 2m) * d^2 psi/dx^2 = E * psi

2. Solucao geral: psi(x) = A*sin(kx) + B*cos(kx)

3. Condicoes de contorno: psi(0) = 0 e psi(L) = 0
   - psi(0) = 0 => B = 0
   - psi(L) = 0 => sin(kL) = 0 => kL = n*pi, com n = 0, 1, 2, ...

4. Portanto k_n = n*pi/L

5. Autoenergias: E_n = hbar^2 * k_n^2 / (2m) = n^2 * pi^2 * hbar^2 / (2mL^2)

6. Normalizacao: integral de 0 a L de |psi|^2 dx = 1
   => A = sqrt(1/L)

Encontrei dois erros propositais acima. Voce consegue identifica-los?
"""

verificacao = ask(solucao_errada, system=SYSTEM_TUTOR, model=SONNET, max_tokens=2048)
print(verificacao)

**Erros inseridos propositalmente:**
1. No passo 3, `n` comeca de **0**, mas `n=0` daria `psi=0` (funcao nula, sem significado fisico). O correto e `n = 1, 2, 3, ...`
2. No passo 6, a constante de normalizacao correta e `A = sqrt(2/L)`, nao `sqrt(1/L)`

O Claude deve identificar ambos e explicar por que sao erros.

---

## Eletromagnetismo: Campos e Potenciais

Vamos pedir ao Claude para derivar o campo eletrico de uma distribuicao de cargas
usando duas abordagens diferentes e comparar os resultados.

In [ ]:
problema_em = """
Considere uma esfera solida de raio R com densidade volumetrica de carga
uniforme rho.

Derive o campo eletrico E(r) para:
a) r < R (dentro da esfera)
b) r > R (fora da esfera)

Use DUAS abordagens:
1. Lei de Gauss (com justificativa da escolha da superficie gaussiana)
2. Integracao direta da lei de Coulomb (pelo menos para o caso r > R)

Compare os resultados e comente sobre a continuidade do campo em r = R.
"""

resposta_em = ask(problema_em, system=SYSTEM_TUTOR, model=SONNET, max_tokens=3000)
print(resposta_em)

**Pontos-chave esperados na resposta:**
- Para r < R: `E = rho * r / (3 * epsilon_0)` (campo cresce linearmente)
- Para r > R: `E = rho * R^3 / (3 * epsilon_0 * r^2)` (equivale a carga pontual)
- Continuidade em r = R: ambas expressoes dao o mesmo valor
- A lei de Gauss e muito mais eficiente para esse caso devido a simetria esferica

---

## Termodinamica: Ciclos e Eficiencia

Aqui demonstramos uma **conversa multi-turno** sobre o ciclo de Carnot,
usando a API de mensagens diretamente para manter o contexto.

In [ ]:
# Conversa multi-turno sobre ciclo de Carnot
mensagens = [
    {
        'role': 'user',
        'content': (
            'Um motor de Carnot opera entre um reservatorio quente a Th = 600 K '
            'e um reservatorio frio a Tc = 300 K. '
            'Calcule a eficiencia do ciclo e explique seu significado fisico.'
        )
    }
]

resp1 = client.messages.create(
    model=SONNET,
    max_tokens=1024,
    system=SYSTEM_TUTOR,
    messages=mensagens
)
texto1 = resp1.content[0].text
print('=== Resposta 1 (Eficiencia) ===')
print(texto1)
print()

In [ ]:
# Pergunta de acompanhamento: e se mudarmos a temperatura?
mensagens.append({'role': 'assistant', 'content': texto1})
mensagens.append({
    'role': 'user',
    'content': (
        'Agora, se eu aumentar Th para 900 K mantendo Tc = 300 K, '
        'como muda a eficiencia? E se eu quisesse eficiencia de 100%%, '
        'o que precisaria acontecer? Relacione com a Segunda Lei.'
    )
})

resp2 = client.messages.create(
    model=SONNET,
    max_tokens=1024,
    system=SYSTEM_TUTOR,
    messages=mensagens
)
texto2 = resp2.content[0].text
print('=== Resposta 2 (Analise da Segunda Lei) ===')
print(texto2)

In [ ]:
# Terceira pergunta: diagrama P-V
mensagens.append({'role': 'assistant', 'content': texto2})
mensagens.append({
    'role': 'user',
    'content': (
        'Descreva em detalhe as 4 etapas do ciclo de Carnot no diagrama P-V. '
        'Para cada etapa, indique se e isotermica ou adiabatica, '
        'qual o trabalho e calor envolvidos, e a direcao no diagrama.'
    )
})

resp3 = client.messages.create(
    model=SONNET,
    max_tokens=1500,
    system=SYSTEM_TUTOR,
    messages=mensagens
)
print('=== Resposta 3 (Etapas do Ciclo) ===')
print(resp3.content[0].text)

**Nota:** A conversa multi-turno permite aprofundar o tema progressivamente.
O Claude mantem o contexto das respostas anteriores, o que e ideal para
sessoes de estudo interativas.

---

## Resolvendo Problemas Passo a Passo com Chain of Thought

A tecnica de **Chain of Thought (CoT)** e especialmente util para derivacoes
longas. Usamos tags XML para estruturar o raciocinio e garantir que
nenhuma etapa seja pulada.

In [ ]:
SYSTEM_COT = """
Voce e um professor de Fisica Teorica. Ao resolver problemas complexos,
estruture sua resposta usando estas tags XML:

<premissas>Identifique hipoteses e condicoes iniciais</premissas>
<desenvolvimento>Mostre cada passo matematico com justificativa fisica</desenvolvimento>
<verificacao>Verifique limites, unidades e casos especiais</verificacao>
<resultado>Apresente o resultado final e sua interpretacao</resultado>

Seja rigoroso e nao pule etapas.
""".strip()

In [ ]:
problema_cot = """
Derive a equacao de Schrodinger para o atomo de hidrogenio (problema de
dois corpos reduzido ao problema de um corpo com massa reduzida).

Mostre:
1. A separacao em coordenadas esfericas (r, theta, phi)
2. A separacao da parte radial e angular
3. Os numeros quanticos que emergem de cada equacao
4. A expressao final para os niveis de energia E_n

Use a estrutura XML definida no system prompt.
"""

derivacao = ask(problema_cot, system=SYSTEM_COT, model=SONNET, max_tokens=4096)
print(derivacao)

**Por que usar tags XML?**

- **Estrutura clara**: Cada secao tem um proposito definido
- **Reproducibilidade**: Facilita extrair partes especificas da resposta via parsing
- **Completude**: O modelo tende a preencher todas as secoes, evitando pular etapas
- **Verificacao**: A tag `<verificacao>` forca o modelo a checar o proprio resultado

In [ ]:
# Exemplo: extrair apenas o resultado final
import re

match = re.search(r'<resultado>(.*?)</resultado>', derivacao, re.DOTALL)
if match:
    print('=== Resultado Extraido ===')
    print(match.group(1).strip())
else:
    print('Tag <resultado> nao encontrada na resposta.')

---

## Exercicios Propostos

1. **Mecanica Quantica**: Use o Claude para resolver o oscilador harmonico quantico.
   Peca que ele derive os niveis de energia `E_n = hbar*omega*(n + 1/2)` passo a passo.

2. **Eletromagnetismo**: Envie um problema sobre a lei de Biot-Savart para um fio
   retilineo infinito e compare com o resultado usando a lei de Ampere.

3. **Termodinamica**: Implemente uma conversa multi-turno sobre o ciclo Otto
   (motor a combustao) e compare a eficiencia com o ciclo de Carnot.

4. **Verificacao de Solucoes**: Escreva uma solucao do efeito fotoeletrico com
   3 erros conceituais e peca ao Claude para encontra-los.

5. **Chain of Thought**: Use as tags XML para pedir a derivacao das equacoes
   de Maxwell na forma diferencial a partir da forma integral.

In [ ]:
# Espaco para resolver os exercicios


---

## Proximos Passos

- **`02_revisao_literatura.ipynb`** - Use o Claude para analisar e resumir artigos cientificos
- **`03_simulacoes_numericas.ipynb`** - Combine Claude com NumPy/SciPy para simulacoes fisicas
- Explore o uso de **extended thinking** (pensamento estendido) para derivacoes ainda mais complexas
- Experimente diferentes **system prompts** para adaptar o nivel de detalhe das explicacoes

---

*Notebook do modulo `10_engfis_academico` -- Engenharia Fisica, UFRGS*